<a href="https://colab.research.google.com/github/arun21733/climate-change/blob/main/milestone3_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

# Page configuration
st.set_page_config(
    page_title="ClimateScope - Climate Intelligence Dashboard (Clone)",
    layout="wide",
    page_icon="🌍"
)

st.title("🌍 ClimateScope Climate Intelligence Dashboard")
st.markdown("""
Interactive dashboard to explore **global climate trends, weather patterns,
and extreme weather events** — recreated from Infosys ClimateScope PR #5.
""")

# Load data (replace with your own CSV path)
# Expected columns: date, country, temperature_celsius, humidity, wind_kph, precipitation_mm, ...
@st.cache_data
def load_data():
    # For real use → pd.read_csv("/content/global-weather-repository/GlobalWeatherRepository.csv")
    # Demo: generating synthetic data that mimics realistic weather
    dates = pd.date_range(start="2020-01-01", end="2025-12-31", freq="D")
    countries = ["India", "USA", "Brazil", "Germany", "Australia", "China", "Japan", "South Africa"]
    np.random.seed(42)
    data = []
    for country in countries:
        temps = 15 + 15 * np.sin(np.linspace(0, 10 * np.pi, len(dates))) + np.random.normal(0, 3, len(dates))
        humids = 60 + 20 * np.random.normal(0, 1, len(dates))
        winds = 10 + 8 * np.random.normal(0, 1, len(dates))
        precips = np.maximum(0, 5 + 15 * np.random.exponential(0.4, len(dates)) - 0.3 * temps)

        df_country = pd.DataFrame({
            "date": dates,
            "country": country,
            "temperature_celsius": temps,
            "humidity": humids,
            "wind_kph": winds,
            "precipitation_mm": precips
        })
        data.append(df_country)

    df = pd.concat(data, ignore_index=True)
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    return df.dropna()
df = load_data()

2026-03-15 08:59:26.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:59:26.004 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:59:26.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:59:26.007 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:59:26.010 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:59:26.010 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:59:26.011 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:59:26.014 No runtime found, using MemoryCacheStorageManager


In [6]:
!pip install opendatasets
import opendatasets as od
od.download("https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: arun24
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository


100%|██████████| 10.3M/10.3M [00:00<00:00, 868MB/s]


In [13]:
# ────────────────────────────────────────────────
# Sidebar Filters (exact match to PR)
# ────────────────────────────────────────────────
st.sidebar.header("Dashboard Filters")

metric = st.sidebar.selectbox(
    "Select Climate Metric",
    options=[
        "temperature_celsius",
        "humidity",
        "wind_kph",
        "precipitation_mm"
    ],
    index=0
)

countries = sorted(df["country"].unique())
selected_countries = st.sidebar.multiselect(
    "Select Countries",
    options=countries,
    default=countries[:4]
    )

date_range = st.sidebar.date_input(
    "Select Date Range",
    value=(df["date"].min(), df["date"].max()),
    min_value=df["date"].min(),
    max_value=df["date"].max()
)

# Apply filters
filtered_df = df[
    (df["country"].isin(selected_countries)) &
    (df["date"] >= pd.to_datetime(date_range[0])) &
    (df["date"] <= pd.to_datetime(date_range[1]))
]

if filtered_df.empty:
    st.error("No data available for the selected filters. Try different selections.")
    st.stop()

# ────────────────────────────────────────────────
# KPI Cards (4-column layout)
# ────────────────────────────────────────────────
st.subheader("Climate Summary")
k1, k2, k3, k4 = st.columns(4)

k1.metric("Avg Temperature", f"{filtered_df['temperature_celsius'].mean():.1f} °C")
k2.metric("Avg Humidity", f"{filtered_df['humidity'].mean():.1f} %")
k3.metric("Avg Wind Speed", f"{filtered_df['wind_kph'].mean():.1f} km/h")
k4.metric("Avg Rainfall", f"{filtered_df['precipitation_mm'].mean():.1f} mm")


2026-03-15 08:40:36.618 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:40:36.619 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:40:36.621 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:40:36.622 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:40:36.623 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:40:36.626 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:40:36.628 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:40:36.630 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [14]:
# ────────────────────────────────────────────────
# Tabs (same structure as original PR)
# ────────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Overview",
    "Temperature Analysis",
    "Climate Relationships",
    "Extreme Events",
    "Global Insights"
])

2026-03-15 08:41:11.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:41:11.003 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:41:11.004 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:41:11.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:41:11.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:41:11.007 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [15]:
# Tab 1: Overview
with tab1:
    st.subheader("Climate Trend by Country")
    fig1 = px.line(
        filtered_df,
        x="date",
        y=metric,
        color="country",
        markers=True,
        title=f"{metric.replace('_', ' ').title()} Trend"
    )
    st.plotly_chart(fig1, use_container_width=True)

    col_left, col_right = st.columns(2)

    with col_left:
        st.subheader("Top 10 Hottest Countries")
        hottest = (
            filtered_df.groupby("country")["temperature_celsius"]
            .mean()
            .sort_values(ascending=False)
            .head(10)
            .reset_index()
        )
        fig2 = px.bar(hottest, x="country", y="temperature_celsius",
                      color="temperature_celsius", title="Hottest Countries")
        st.plotly_chart(fig2, use_container_width=True)

    with col_right:
        st.subheader("Top 10 Coldest Countries")
        coldest = (
            filtered_df.groupby("country")["temperature_celsius"]
            .mean()
            .sort_values()
            .head(10)
            .reset_index()
        )
        fig3 = px.bar(coldest, x="country", y="temperature_celsius",
                      color="temperature_celsius", title="Coldest Countries")
        st.plotly_chart(fig3, use_container_width=True)

2026-03-15 08:42:22.850 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:42:22.851 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:42:22.854 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:42:23.505 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-15 08:42:23.690 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:42:23.691 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:42:23.698 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [16]:
# Tab 2: Temperature Analysis
with tab2:
    st.subheader("Temperature Distribution")
    fig4 = px.histogram(filtered_df, x="temperature_celsius", nbins=50,
                        title="Temperature Distribution")
    st.plotly_chart(fig4, use_container_width=True)

    st.subheader("Monthly Temperature Trend")
    monthly = filtered_df.groupby("month")["temperature_celsius"].mean().reset_index()
    fig5 = px.line(monthly, x="month", y="temperature_celsius", markers=True)
    st.plotly_chart(fig5, use_container_width=True)

    st.subheader("Country Temperature Boxplot")
    fig6 = px.box(filtered_df, x="country", y="temperature_celsius")
    st.plotly_chart(fig6, use_container_width=True)

    st.subheader("7-Day Rolling Average Temperature")
    rolling = (
        filtered_df.groupby("date")["temperature_celsius"]
        .mean()
        .rolling(window=7, min_periods=1)
        .mean()
        .reset_index()
    )
    fig7 = px.line(rolling, x="date", y="temperature_celsius")
    st.plotly_chart(fig7, use_container_width=True)

2026-03-15 08:43:06.858 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:06.860 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:06.861 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:06.938 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-15 08:43:06.941 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:06.942 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:06.943 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [17]:
# Tab 3: Climate Relationships
with tab3:
    st.subheader("Temperature vs Humidity")
    fig8 = px.scatter(filtered_df, x="temperature_celsius", y="humidity",
                      color="country", hover_name="country")
    st.plotly_chart(fig8, use_container_width=True)

    st.subheader("Humidity vs Rainfall")
    fig9 = px.scatter(filtered_df, x="humidity", y="precipitation_mm",
                      color="country")
    st.plotly_chart(fig9, use_container_width=True)

    st.subheader("Climate Correlation Heatmap")
    numeric_cols = filtered_df.select_dtypes(include=np.number).columns
    corr = filtered_df[numeric_cols].corr()
    fig10 = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r")
    st.plotly_chart(fig10, use_container_width=True)

2026-03-15 08:43:37.769 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:37.770 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:37.772 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:37.817 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-15 08:43:37.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:37.832 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:43:37.834 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [18]:
# Tab 4: Extreme Events
with tab4:
    st.subheader("Extreme Temperature Events")
    heat_threshold = filtered_df["temperature_celsius"].quantile(0.95)
    extreme_heat = filtered_df[filtered_df["temperature_celsius"] > heat_threshold]
    st.metric("Extreme Heat Events (95th percentile)", len(extreme_heat))

    fig11 = px.scatter(extreme_heat, x="date", y="temperature_celsius",
                       color="country", title="Extreme Heat Events")
    st.plotly_chart(fig11, use_container_width=True)

    st.subheader("Potential Flood Risk Periods")
    rain_threshold = filtered_df["precipitation_mm"].quantile(0.95)
    flood_risk = filtered_df[filtered_df["precipitation_mm"] > rain_threshold]
    st.metric("High Rainfall Events (95th percentile)", len(flood_risk))

    fig12 = px.scatter(flood_risk, x="date", y="precipitation_mm",
                       color="country", title="High Rainfall / Flood Risk Events")
    st.plotly_chart(fig12, use_container_width=True)

2026-03-15 08:45:48.363 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:45:48.365 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:45:48.366 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:45:48.373 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:45:48.373 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:45:48.374 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:45:48.416 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


In [19]:
# Tab 5: Global Insights (placeholder — extend as needed)
with tab5:
    st.subheader("Global Insights (to be expanded)")
    st.markdown("""
    - Average global temperature trend
    - Year-over-year changes
    - Country rankings over time
    - Climate anomaly detection
    """)
    st.info("This tab was partially implemented in the original PR — add more visualizations here.")
    # Climate Intelligence Dashboard
# Interactive visualization of global weather and climate trends

2026-03-15 08:57:40.175 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:57:40.177 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:57:40.178 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:57:40.180 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:57:40.181 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:57:40.182 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:57:40.184 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-15 08:57:40.185 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [24]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

# Page configuration
st.set_page_config(
    page_title="ClimateScope - Climate Intelligence Dashboard (Clone)",
    layout="wide",
    page_icon="🌍"
)

st.title("🌍 ClimateScope Climate Intelligence Dashboard")
st.markdown("""
Interactive dashboard to explore **global climate trends, weather patterns,
and extreme weather events** — recreated from Infosys ClimateScope PR #5.
""")

# Load data (replace with your own CSV path)
# Expected columns: date, country, temperature_celsius, humidity, wind_kph, precipitation_mm, ...
@st.cache_data
def load_data():
    # For real use → pd.read_csv("/content/global-weather-repository/GlobalWeatherRepository.csv")
    # Demo: generating synthetic data that mimics realistic weather
    dates = pd.date_range(start="2020-01-01", end="2025-12-31", freq="D")
    countries = ["India", "USA", "Brazil", "Germany", "Australia", "China", "Japan", "South Africa"]
    np.random.seed(42)
    data = []
    for country in countries:
        temps = 15 + 15 * np.sin(np.linspace(0, 10 * np.pi, len(dates))) + np.random.normal(0, 3, len(dates))
        humids = 60 + 20 * np.random.normal(0, 1, len(dates))
        winds = 10 + 8 * np.random.normal(0, 1, len(dates))
        precips = np.maximum(0, 5 + 15 * np.random.exponential(0.4, len(dates)) - 0.3 * temps)

        df_country = pd.DataFrame({
            "date": dates,
            "country": country,
            "temperature_celsius": temps,
            "humidity": humids,
            "wind_kph": winds,
            "precipitation_mm": precips
        })
        data.append(df_country)

    df = pd.concat(data, ignore_index=True)
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    return df.dropna()
df = load_data()

# Sidebar Filters (exact match to PR)
st.sidebar.header("Dashboard Filters")

metric = st.sidebar.selectbox(
    "Select Climate Metric",
    options=[
        "temperature_celsius",
        "humidity",
        "wind_kph",
        "precipitation_mm"
    ],
    index=0
)

countries = sorted(df["country"].unique())
selected_countries = st.sidebar.multiselect(
    "Select Countries",
    options=countries,
    default=countries[:4]
)

date_range = st.sidebar.date_input(
    "Select Date Range",
    value=(df["date"].min(), df["date"].max()),
    min_value=df["date"].min(),
    max_value=df["date"].max()
)

# Apply filters
filtered_df = df[
    (df["country"].isin(selected_countries)) &
    (df["date"] >= pd.to_datetime(date_range[0])) &
    (df["date"] <= pd.to_datetime(date_range[1]))
]

if filtered_df.empty:
    st.error("No data available for the selected filters. Try different selections.")
    st.stop()

# KPI Cards (4-column layout)
st.subheader("Climate Summary")
k1, k2, k3, k4 = st.columns(4)

k1.metric("Avg Temperature", f"{filtered_df['temperature_celsius'].mean():.1f} °C")
k2.metric("Avg Humidity", f"{filtered_df['humidity'].mean():.1f} %")
k3.metric("Avg Wind Speed", f"{filtered_df['wind_kph'].mean():.1f} km/h")
k4.metric("Avg Rainfall", f"{filtered_df['precipitation_mm'].mean():.1f} mm")

# Tabs (same structure as original PR)
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Overview",
    "Temperature Analysis",
    "Climate Relationships",
    "Extreme Events",
    "Global Insights"
])

# Tab 1: Overview
with tab1:
    st.subheader("Climate Trend by Country")
    fig1 = px.line(
        filtered_df,
        x="date",
        y=metric,
        color="country",
        markers=True,
        title=f"{metric.replace('_', ' ').title()} Trend"
    )
    st.plotly_chart(fig1, use_container_width=True)

    col_left, col_right = st.columns(2)

    with col_left:
        st.subheader("Top 10 Hottest Countries")
        hottest = (
            filtered_df.groupby("country")["temperature_celsius"]
            .mean()
            .sort_values(ascending=False)
            .head(10)
            .reset_index()
        )
        fig2 = px.bar(hottest, x="country", y="temperature_celsius",
                      color="temperature_celsius", title="Hottest Countries")
        st.plotly_chart(fig2, use_container_width=True)

    with col_right:
        st.subheader("Top 10 Coldest Countries")
        coldest = (
            filtered_df.groupby("country")["temperature_celsius"]
            .mean()
            .sort_values()
            .head(10)
            .reset_index()
        )
        fig3 = px.bar(coldest, x="country", y="temperature_celsius",
                      color="temperature_celsius", title="Coldest Countries")
        st.plotly_chart(fig3, use_container_width=True)

# Tab 2: Temperature Analysis
with tab2:
    st.subheader("Temperature Distribution")
    fig4 = px.histogram(filtered_df, x="temperature_celsius", nbins=50,
                        title="Temperature Distribution")
    st.plotly_chart(fig4, use_container_width=True)

    st.subheader("Monthly Temperature Trend")
    monthly = filtered_df.groupby("month")["temperature_celsius"].mean().reset_index()
    fig5 = px.line(monthly, x="month", y="temperature_celsius", markers=True)
    st.plotly_chart(fig5, use_container_width=True)

    st.subheader("Country Temperature Boxplot")
    fig6 = px.box(filtered_df, x="country", y="temperature_celsius")
    st.plotly_chart(fig6, use_container_width=True)

    st.subheader("7-Day Rolling Average Temperature")
    rolling = (
        filtered_df.groupby("date")["temperature_celsius"]
        .mean()
        .rolling(window=7, min_periods=1)
        .mean()
        .reset_index()
    )
    fig7 = px.line(rolling, x="date", y="temperature_celsius")
    st.plotly_chart(fig7, use_container_width=True)

# Tab 3: Climate Relationships
with tab3:
    st.subheader("Temperature vs Humidity")
    fig8 = px.scatter(filtered_df, x="temperature_celsius", y="humidity",
                      color="country", hover_name="country")
    st.plotly_chart(fig8, use_container_width=True)

    st.subheader("Humidity vs Rainfall")
    fig9 = px.scatter(filtered_df, x="humidity", y="precipitation_mm",
                      color="country")
    st.plotly_chart(fig9, use_container_width=True)

    st.subheader("Climate Correlation Heatmap")
    numeric_cols = filtered_df.select_dtypes(include=np.number).columns
    corr = filtered_df[numeric_cols].corr()
    fig10 = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r")
    st.plotly_chart(fig10, use_container_width=True)

# Tab 4: Extreme Events
with tab4:
    st.subheader("Extreme Temperature Events")
    heat_threshold = filtered_df["temperature_celsius"].quantile(0.95)
    extreme_heat = filtered_df[filtered_df["temperature_celsius"] > heat_threshold]
    st.metric("Extreme Heat Events (95th percentile)", len(extreme_heat))

    fig11 = px.scatter(extreme_heat, x="date", y="temperature_celsius",
                       color="country", title="Extreme Heat Events")
    st.plotly_chart(fig11, use_container_width=True)

    st.subheader("Potential Flood Risk Periods")
    rain_threshold = filtered_df["precipitation_mm"].quantile(0.95)
    flood_risk = filtered_df[filtered_df["precipitation_mm"] > rain_threshold]
    st.metric("High Rainfall Events (95th percentile)", len(flood_risk))

    fig12 = px.scatter(flood_risk, x="date", y="precipitation_mm",
                       color="country", title="High Rainfall / Flood Risk Events")
    st.plotly_chart(fig12, use_container_width=True)

# Tab 5: Global Insights (placeholder — extend as needed)
with tab5:
    st.subheader("Global Insights (to be expanded)")
    st.markdown("""
    - Average global temperature trend
    - Year-over-year changes
    - Country rankings over time
    - Climate anomaly detection
    """)
    st.info("This tab was partially implemented in the original PR — add more visualizations here.")
# Interactive visualization of global weather and climate trends

Overwriting app.py


In [ ]:
!pip install streamlit
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.86.68.117:8501

